# Train a tiny diffusion model

The data are points on a thin ring of radius 2. You will build a noise schedule that really reaches pure noise, train a noise-predicting network with a time embedding, generate points with the 100-step ancestral sampler, and then with DDIM in as few as 5 steps.

**How to work through it.** Each coding part gives you a function to write:
its name, what goes in and what must come out. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under
it. The check prints `Part N passed` when your answer is right and raises
an error that says what is wrong when it is not.

## Setup

Run this cell first. It loads the libraries and the data every part uses.

In [ ]:
import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

T = 100                                            # number of noise levels


def sample_data(n, generator=None):
    """n points on a thin ring of radius 2: the distribution the model must learn."""
    theta = 2 * math.pi * torch.rand(n, generator=generator)
    r = 2.0 + 0.05 * torch.randn(n, generator=generator)
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1)


def ring_score(samples):
    """Fraction of samples whose radius is within 0.2 of 2."""
    return ((samples.norm(dim=1) - 2.0).abs() < 0.2).float().mean().item()


data = sample_data(2000, torch.Generator().manual_seed(0))
print("ring score of real data:", ring_score(data))

## Part 1 (coding task): A schedule that really reaches noise

The forward process mixes data with noise: $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol\epsilon$ with $\boldsymbol\epsilon \sim \mathcal{N}(\mathbf{0}, I)$, where $\bar\alpha_t = \prod_{s \le t}(1 - \beta_s)$. Sampling later starts from pure noise, so $\bar\alpha$ at the last step must be close to 0.

- `linear_alpha_bar(T, beta_1=1e-4, beta_T=0.02)`: $\bar\alpha_t$ for $\beta$ spaced evenly from `beta_1` to `beta_T`, with `torch.cumprod`.
- `cosine_schedule(T, s=0.008)` returns `(betas, alphas, alpha_bar)`, each of length `T`. Define $f(t) = \cos^2\!\big(\tfrac{t/T + s}{1 + s}\cdot\tfrac{\pi}{2}\big)$ for $t = 0, \dots, T$, set $\beta_t = 1 - f(t)/f(t-1)$ for $t = 1, \dots, T$, clamp it at 0.999, and recompute $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t$ with `torch.cumprod`.
- `q_sample(x0, t, eps)` returns $\mathbf{x}_t$ for a batch, with `t` a tensor of step indices from 0 to `T - 1` (index `t` is noise level $t + 1$).

Print $\bar\alpha$ at the last step for both schedules.

In [ ]:
def linear_alpha_bar(T, beta_1=1e-4, beta_T=0.02):
    # TODO
    raise NotImplementedError


def cosine_schedule(T, s=0.008):
    # TODO
    raise NotImplementedError


betas, alphas, alpha_bar = cosine_schedule(T)


def q_sample(x0, t, eps):
    # TODO
    raise NotImplementedError

In [ ]:
assert betas.shape == alphas.shape == alpha_bar.shape == (T,), "three length-T tensors"
assert (alpha_bar[1:] < alpha_bar[:-1]).all() and alpha_bar[0] > 0.999, "alpha_bar starts near 1 and decreases"
assert alpha_bar[-1] < 1e-3, "the cosine schedule should end in (almost) pure noise"
assert abs(linear_alpha_bar(T)[-1].item() - 0.3636) < 1e-3, "the linear schedule over 100 steps ends at 0.36"
x0 = torch.randn(4096, 2)
eps = torch.randn(4096, 2)
for step in (0, T // 2, T - 1):
    xt = q_sample(x0, torch.full((4096,), step), eps)
    assert abs(xt.std().item() - 1.0) < 0.05, f"q_sample should keep unit variance at step {step}"
assert torch.allclose(q_sample(x0, torch.zeros(4096, dtype=torch.long), torch.zeros_like(eps)), alpha_bar[0].sqrt() * x0), "with eps = 0, x_t is sqrt(alpha_bar) x0"
print("Part 1 passed")

## Part 2 (coding task): A noise predictor that knows the time

The model $\boldsymbol\epsilon_\theta(\mathbf{x}_t, t)$ predicts the noise that was mixed in. It needs to know $t$, because at small $t$ there is little noise to find and at large $t$ there is little data left.

- `timestep_embedding(t, dim=16)` turns integer steps into vectors: with $k = 0, \dots, \tfrac{dim}{2} - 1$ and frequencies $\omega_k = 1000^{-2k/dim}$, return $[\sin(\omega_k t)]_k$ followed by $[\cos(\omega_k t)]_k$, shape `(len(t), dim)`.
- `EpsModel(nn.Module)` concatenates $\mathbf{x}_t$ with the embedding (2 + 16 inputs) and applies `Linear(18, 128)`, `SiLU`, `Linear(128, 128)`, `SiLU`, `Linear(128, 128)`, `SiLU`, `Linear(128, 2)`.
- `train_ddpm(model, steps=3000, batch=256, lr=1e-3, seed=0)` trains with Adam on the simplified loss: draw fresh ring points (from `sample_data` with a generator seeded with `seed`), uniform steps `t`, and noise $\boldsymbol\epsilon$, and minimise $\lVert\boldsymbol\epsilon - \boldsymbol\epsilon_\theta(\mathbf{x}_t, t)\rVert^2$, averaged. Return the list of losses.

Part 2 trains for 3000 steps and is the only slow part of the notebook: the whole notebook takes about a minute and a half on a laptop CPU, and no GPU or download is needed.

In [ ]:
def timestep_embedding(t, dim=16):
    # TODO
    raise NotImplementedError


class EpsModel(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO

    def forward(self, x, t):
        # TODO
        raise NotImplementedError


def train_ddpm(model, steps=3000, batch=256, lr=1e-3, seed=0):
    # TODO
    raise NotImplementedError


torch.manual_seed(0)
model = EpsModel()
losses = train_ddpm(model)

In [ ]:
emb = timestep_embedding(torch.tensor([0, 7, 99]))
assert emb.shape == (3, 16), "one 16-number vector per step"
assert torch.allclose(emb[0], torch.cat([torch.zeros(8), torch.ones(8)])), "t = 0 gives sin = 0 and cos = 1"
assert torch.allclose(emb[1, 0], torch.sin(torch.tensor(7.0))), "the first frequency is 1"
assert model(torch.randn(5, 2), torch.randint(0, T, (5,))).shape == (5, 2), "the model predicts a 2-D noise vector"
x = torch.randn(5, 2)
assert not torch.allclose(model(x, torch.zeros(5, dtype=torch.long)), model(x, torch.full((5,), T - 1))), "the model's output must depend on t"
late = sum(losses[-200:]) / 200
print(f"mean loss over the last 200 steps: {late:.3f}")
assert late < 0.6 and late > 0.2, "the loss should settle between 0.2 and 0.6"
print("Part 2 passed")

## Part 3 (coding task): Ancestral sampling, 100 steps

Write `ddpm_sample(model, n)`. Start from $\mathbf{x} \sim \mathcal{N}(\mathbf{0}, I)$ and go **backwards** through the steps $t = T-1, \dots, 0$:

$$
\boldsymbol\mu = \frac{1}{\sqrt{\alpha_t}}\Big(\mathbf{x} - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\,\boldsymbol\epsilon_\theta(\mathbf{x}, t)\Big),
\qquad
\mathbf{x} \leftarrow \boldsymbol\mu + \sqrt{\beta_t}\,\mathbf{z}\ \ (\mathbf{z} \sim \mathcal{N}(\mathbf{0}, I)),
$$

except at the last step, $t = 0$, where $\mathbf{x} \leftarrow \boldsymbol\mu$ with no added noise. Work under `torch.no_grad()`. Generate 2,000 points after `torch.manual_seed(1)`, print the mean and standard deviation of their radius and their `ring_score`, and plot them over the real data.

In [ ]:
def ddpm_sample(model, n):
    # TODO
    raise NotImplementedError


torch.manual_seed(1)
ddpm_points = ddpm_sample(model, 2000)
# TODO: print the radius statistics and the ring score, and plot

In [ ]:
assert ddpm_points.shape == (2000, 2) and torch.isfinite(ddpm_points).all(), "2000 finite 2-D points"
assert abs(ddpm_points.norm(dim=1).mean().item() - 2.0) < 0.15, "the samples should sit on the ring of radius 2"
assert ring_score(ddpm_points) > 0.85, "most samples should be within 0.2 of the ring"
print("Part 3 passed")

## Part 4 (coding task): DDIM: the same model, five times fewer steps

DDIM reuses the trained model but jumps between a subset of the steps deterministically. At a step $t$ the model's noise estimate gives an estimate of the clean point,

$$
\hat{\mathbf{x}}_0 = \frac{\mathbf{x} - \sqrt{1 - \bar\alpha_t}\,\boldsymbol\epsilon_\theta(\mathbf{x}, t)}{\sqrt{\bar\alpha_t}},
$$

and DDIM moves straight to the next chosen step $t'$ by re-noising that estimate with the **same** noise: $\mathbf{x} \leftarrow \sqrt{\bar\alpha_{t'}}\,\hat{\mathbf{x}}_0 + \sqrt{1 - \bar\alpha_{t'}}\,\hat{\boldsymbol\epsilon}$.

Write `ddim_sample(model, n, steps, clip=3.0)`. Use the steps `torch.linspace(T - 1, 0, steps).round().long()`. At each one, compute $\hat{\mathbf{x}}_0$, clamp it to $[-\texttt{clip}, \texttt{clip}]$, recompute $\hat{\boldsymbol\epsilon} = (\mathbf{x} - \sqrt{\bar\alpha_t}\,\hat{\mathbf{x}}_0)/\sqrt{1 - \bar\alpha_t}$ so that the two stay consistent, and jump to the next step. At the last step return $\hat{\mathbf{x}}_0$. Store the ring score of 2,000 samples in `ddim_scores[steps]` for 50, 20, 10 and 5 steps, each with a generator seeded with 1.

In [ ]:
def ddim_sample(model, n, steps, clip=3.0, generator=None):
    # TODO
    raise NotImplementedError


ddim_scores = {}
# TODO

In [ ]:
assert set(ddim_scores) == {50, 20, 10, 5}, "store a score for 50, 20, 10 and 5 steps"
assert ddim_scores[20] > 0.85, "20 DDIM steps should do about as well as 100 DDPM steps"
assert ddim_scores[5] < ddim_scores[20], "with only 5 steps quality should start to drop"
a = ddim_sample(model, 50, 20, generator=torch.Generator().manual_seed(1))
b = ddim_sample(model, 50, 20, generator=torch.Generator().manual_seed(1))
assert torch.equal(a, b), "given the starting noise, DDIM is deterministic"
print("Part 4 passed")

## Solution

### Part 1: A schedule that really reaches noise

In [ ]:
def linear_alpha_bar(T, beta_1=1e-4, beta_T=0.02):
    return torch.cumprod(1 - torch.linspace(beta_1, beta_T, T), dim=0)


def cosine_schedule(T, s=0.008):
    t = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos((t / T + s) / (1 + s) * math.pi / 2) ** 2
    betas = (1 - f[1:] / f[:-1]).clamp(max=0.999).float()
    alphas = 1 - betas
    return betas, alphas, torch.cumprod(alphas, dim=0)


betas, alphas, alpha_bar = cosine_schedule(T)


def q_sample(x0, t, eps):
    ab = alpha_bar[t].unsqueeze(1)                 # one value per example, broadcast over x and y
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps


print(f"last-step alpha_bar: linear {linear_alpha_bar(T)[-1]:.4f}, cosine {alpha_bar[-1]:.2e}")

**Why it works.** With the linear schedule borrowed from the 1000-step DDPM paper but only 100 steps, $\bar\alpha_T = 0.36$, so the "noisiest" point still keeps $\sqrt{0.36} = 60\%$ of the data, and a sampler that starts from pure noise starts from a distribution the model never saw. The cosine schedule is designed to bring $\bar\alpha$ down smoothly to almost exactly 0 in any number of steps. Because $(\sqrt{\bar\alpha_t})^2 + (\sqrt{1 - \bar\alpha_t})^2 = 1$, unit-variance data keeps unit variance at every step, so one network can handle every noise level without rescaling its input.

### Part 2: A noise predictor that knows the time

In [ ]:
def timestep_embedding(t, dim=16):
    half = dim // 2
    freqs = 1000.0 ** (-torch.arange(half) / half)              # omega_k = 1000^(-2k/dim)
    angles = t.float()[:, None] * freqs[None, :]
    return torch.cat([angles.sin(), angles.cos()], dim=1)


class EpsModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + 16, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, timestep_embedding(t)], dim=1))


def train_ddpm(model, steps=3000, batch=256, lr=1e-3, seed=0):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    losses = []
    for step in range(steps):
        x0 = sample_data(batch, g)
        t = torch.randint(0, T, (batch,))
        eps = torch.randn_like(x0)
        loss = ((model(q_sample(x0, t, eps), t) - eps) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        if step % 500 == 0:
            print(f"step {step:4d}  loss {loss.item():.4f}")
    return losses


torch.manual_seed(0)
model = EpsModel()
losses = train_ddpm(model)

**Why it works.** The sinusoidal embedding gives every step a distinct vector whose coordinates vary on many time scales, from the fast $\sin t$ to slow ones, which a small network separates more easily than the single number $t/T$. The loss settles well above 0, and that is success: at small $t$ the added noise is tiny and leaves almost no trace in $\mathbf{x}_t$, so it cannot be predicted and the error there stays near 1 per coordinate. Averaged over all $t$ the floor is around 0.4. A loss heading for zero would mean the model had found a way to see the true noise, which for this objective is a bug.

### Part 3: Ancestral sampling, 100 steps

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n):
    x = torch.randn(n, 2)
    for t in reversed(range(T)):
        eps = model(x, torch.full((n,), t))
        mean = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
        x = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean
    return x


torch.manual_seed(1)
ddpm_points = ddpm_sample(model, 2000)
radius = ddpm_points.norm(dim=1)
print(f"radius mean {radius.mean():.3f}, std {radius.std():.3f}, ring score {ring_score(ddpm_points):.3f}")
plt.figure(figsize=(4, 4))
plt.scatter(data[:, 0], data[:, 1], s=2, alpha=0.3, label="data")
plt.scatter(ddpm_points[:, 0], ddpm_points[:, 1], s=2, alpha=0.3, label="DDPM samples")
plt.gca().set_aspect("equal")
plt.legend()
plt.show()

**Why it works.** Each step removes the predicted noise, rescales, and adds back a little fresh noise, so the chain walks from pure noise to the ring through 100 small denoising moves. The fresh noise keeps the samples spread around the whole ring instead of collapsing onto a few points. It is skipped at the final step, because nothing after it would remove it. The ring is thicker than the real one (radius standard deviation about 0.09 against 0.05): small errors in $\boldsymbol\epsilon_\theta$ accumulate over the chain.

### Part 4: DDIM: the same model, five times fewer steps

In [ ]:
@torch.no_grad()
def ddim_sample(model, n, steps, clip=3.0, generator=None):
    schedule = torch.linspace(T - 1, 0, steps).round().long().tolist()
    x = torch.randn(n, 2, generator=generator)
    for i, t in enumerate(schedule):
        eps = model(x, torch.full((n,), t))
        x0 = ((x - (1 - alpha_bar[t]).sqrt() * eps) / alpha_bar[t].sqrt()).clamp(-clip, clip)
        if i == len(schedule) - 1:
            return x0
        eps = (x - alpha_bar[t].sqrt() * x0) / (1 - alpha_bar[t]).sqrt()
        t_next = schedule[i + 1]
        x = alpha_bar[t_next].sqrt() * x0 + (1 - alpha_bar[t_next]).sqrt() * eps


ddim_scores = {}
for steps in (50, 20, 10, 5):
    torch.manual_seed(1)
    ddim_scores[steps] = ring_score(ddim_sample(model, 2000, steps))
    print(f"DDIM with {steps:2d} model calls: ring score {ddim_scores[steps]:.3f}")
print(f"DDPM with {T} model calls: ring score {ring_score(ddpm_points):.3f}")

**Why it works.** DDIM keeps the same training and the same model and changes only the sampler: given the starting noise it is deterministic, and it can skip steps because each update lands exactly on the forward process's marginal at the next chosen step. Twenty model calls do as well as 100 ancestral steps, and quality only drops off at around five. The clamp matters with the cosine schedule: at the first step $\bar\alpha \approx 2 \times 10^{-7}$, so dividing by $\sqrt{\bar\alpha_t}$ multiplies any error in $\boldsymbol\epsilon_\theta$ by about 2,000, and without the clamp the samples fly off far beyond the ring. Clamping to a box that comfortably contains the data, here $[-3, 3]$, is standard practice for image diffusion too, where pixels are clipped to $[-1, 1]$.